*0.3 Classical NLP*

# Tokenization: SentencePiece

**The situation.** Your product launches in Japanese and Thai. Neither language puts spaces between words. The English pipeline's "split on whitespace, then subword" step produces one giant token per sentence. Search and classification stop working on day one.

**SentencePiece.** A tokenizer that treats the input as a raw stream of characters — spaces included, written as `▁` — and learns pieces directly from it, no whitespace step first. That makes it language-independent and exactly reversible: join the pieces, replace `▁` with space, and you get the original text back. Llama, T5 and Gemma use it.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Train one on a mixed English/Japanese corpus** and tokenize both. `sentencepiece` writes a small model file; in production it ships with the model.

In [2]:
import io

import sentencepiece as spm

corpus = [
    "The customer was charged twice for the order.",
    "Please refund the duplicate charge on the invoice.",
    "The refund for the duplicate invoice was processed yesterday.",
    "Cancel my subscription and refund the last payment.",
    "Please cancel the order and refund the customer.",
    "The invoice for the subscription was charged twice.",
    "お客様は注文に対して二重に請求されました。",
    "重複した請求の返金をお願いします。",
    "昨日、重複した請求書の返金が処理されました。",
    "サブスクリプションを解約して、最後の支払いを返金してください。",
    "注文を解約して、お客様に返金してください。",
    "サブスクリプションの請求書が二重に請求されました。",
] * 30

model = io.BytesIO()
spm.SentencePieceTrainer.train(
    sentence_iterator=iter(corpus),
    model_writer=model,
    vocab_size=120,
    model_type="unigram",
    minloglevel=2,
)
tokenizer = spm.SentencePieceProcessor(model_proto=model.getvalue())

for text in ["Please refund the charge.", "返金をお願いします。"]:
    pieces = tokenizer.encode(text, out_type=str)
    print(f"{text!r:<30} → {pieces}")
    assert tokenizer.decode(pieces) == text  # exactly reversible
print("round trip exact: True")

'Please refund the charge.'    → ['▁Please', '▁refund', '▁the', '▁charge', '.']
'返金をお願いします。'                   → ['▁', '返金', 'を', 'お', '願', 'い', 'し', 'ま', 'す', '。']
round trip exact: True


**Reading the output.** English pieces carry a leading `▁` where a space was. The Japanese sentence — no spaces anywhere — still splits into pieces ("返金", *refund*, became one piece from this tiny corpus; the rest stay as characters), because SentencePiece never needed spaces. Both decode back to the exact original.

```
"Please refund"   →  ▁Please  ▁refund          (▁ = the space, kept as data)
"返金をお願い"      →  返金  を  お願い            (no spaces needed to find pieces)
```

**The rule to remember.** SentencePiece works on raw text, so it is the choice for multilingual models and for anything that must reproduce the input exactly.

| Use it when | Don't when | Instead use |
|---|---|---|
| multilingual text; Llama/T5/Gemma family models | you are using a GPT or BERT model | tiktoken / WordPiece |

**Watch out**
- `model_type="unigram"` (the default) and `"bpe"` are both available; they produce different pieces from the same text.
- Leading `▁` means `"refund"` and `" refund"` are different tokens; watch prompt formatting at word boundaries.
- The trained `.model` file must ship with the model; losing it makes the weights useless.